# பாடம் 17 - Foundry Local மற்றும் Qwen உடன் உள்ளூர் AI முகவலர்களை உருவாக்குதல்

இந்த நோட்புக்-இல் நீங்கள் முழுமையாக உங்கள் வேலைநிறுவனத்தில் இயங்கும் **உள்ளூர் பொறியியல் உதவியாளர்** ஒன்றைப் போடுகிறீர்கள். எதாவது நேரத்தில் மேக கணிப்பொறி பயன்படவில்லை. இந்த உதவியாளர் செய்ய முடியும்:

1. **கருவிகள் அழைக்கவும்** Qwen செயல்பாட்டு அழைப்பின் மூலம் Foundry Local மூலம்.
2. **ஒரு பாதுகாக்கப்பட்ட திட்ட அடைவை உள்ளே கோப்புகளை பட்டியலிடவும் மற்றும் படிக்கவும்.**
3. **எளிய அளவுகோல்கள் கொண்டு குறியீட்டை பகுப்பாய்வு செய்யவும்.**
4. **உள்ளூர் RAG (Chroma) மூலம் ஆவணங்களை தேடவும்.**
5. **ஒரு உள்ளூர் MCP சேவையகத்தை பயன்படுத்தவும்** (ஏதும் அமைக்கப்படவில்லை என்றால் மெல்லிசையாகத் தவிர்க்கப்படுகிறது).

இந்த முகவர் குறியீடு மேக பாடங்களுடன் எளிதில் ஒன்று போன்றது — ஒரே வித்தியாசம் வாடிக்கையாளர் நிலைமையம் மேகத்திலிருந்து `localhost`க்கு மாறுகிறது.


## அமைப்பு

இந்த நோட்புக் ஓடுவதற்கு முன்:

1. **Microsoft Foundry Local ஐ நிறுவுக** ([தளவமைவை](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) உங்கள் OSக்கு பார்க்கவும்).
2. **Qwen மாதிரியை பதிவிறக்கம் செய்து தொடங்கு:**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. கீழே உள்ள Python பைசகேஜ்களை நிறுவவும்.

அனைத்தும் உள்ளகமாக இயங்கும். சுமார் 8 GB RAM கொண்ட கணினி நியாயமான குறைந்தபட்சம் ஆகும்.


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## 1. Foundry Local-க்கு இணைவு அமைக்கு

`FoundryLocalManager` தேவையான மாடலை பதிவிறக்கி, உள்ளக சேவையை தொடங்கி, எங்களுக்கு **OpenAI மூலம் இணைக்கக்கூடிய ஒரு இறுதி இடைமுகத்தை** வழங்குகிறது. பின்னர் நாங்கள் பொதுவான OpenAI SDK-ஐ அதனிடத்தில் அமைக்கின்றோம். API விசை ஒரு உள்ளக இடைக்கால குறியீடாகும் — இதில் எந்த மேக அங்கீகாரம் தேவையில்லை.


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## 2. உள்ளூர் கருவிகள் (Sandboxed கோப்பு செயல்பாடுகள்)

நாங்கள் டிஸ்க்கில் ஒரு சிறிய மாதிரி திட்டத்தை உருவாக்குகிறோம், பின்னர் அந்த திட்டத்தின் ரூட் க்கு இருக்கும் கருவிகளை வரையறுக்கிறோம். Sandbox சரிபார்ப்பு உள்ளூரிலும் முக்கியம்: சிக்கலான பாதைகளைக் கையாளும் கருவி உங்கள் பயனர் அனுமதிகளுடன் இயங்கும் மற்றும் நீங்கள் அணுகக்கூடிய எல்லாவற்றையும் தொடலாம்.


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. குறுப்பிடப்பட்ட RAG உடன் உள்ளூர் Chroma

நாங்கள் ஒரு சிறிய ஆவண துண்டுகளின் தொகுப்பை உள்ளூர் Chroma சேகரிப்பில் நுழைக்கிறோம். Chroma செயலியில் இயங்குகிறது மற்றும் வெக்டர் டிஸ்கில் சேமிக்கின்றது — எந்த சர்வரும் இல்லை, எந்த மேகமும் இல்லை. `search_docs` கருவி ஒரு கேள்விக்கான மிகவும் தொடர்புடைய துண்டுகளை மீட்டெடுக்கும்.


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## 4. கருவி-கொல்கின்ற மடக்கு

இப்போது நாம் OpenAI கருவிகள் வாரியத்தைப் பயன்படுத்தி கருவிகளை மாதிரியில் பதிவு செய்து, முறைப்படி கருவி-கொல்கின்ற மடக்கை இயக்குகிறோம் — மாதிரி ஒரு கருவியை கோருகின்றது, நாங்கள் அதை உள்ளூராக செயல்படுத்துகிறோம், முடிவை மீண்டும் ஈட்டுகிறோம், மற்றும் மாதிரி இறுதி பதிலை உருவாக்கும் வரை மீண்டும் செய்கிறோம். Qwen-இன் நம்பகமான செயல்பாடு அழைத்தல் தான் இதை சாதனத்தில் இயங்க வைக்கிறது.


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## 5. உள்ளமைந்த MCP (விருப்பமானது)

MCP என்பது ஒரு மேக சேவை அல்ல, ஒரு போக்குவரத்து முறையாகும் — ஒரு MCP சர்வர் உள்ளமைந்த செயல்முறையாக `stdio` மூலம் இயங்கலாம். கீழுள்ள செலல் உங்கள் உள்ளமைந்த MCP சர்வருடன் (உதாரணமாக ஒரு கோப்பு முறைநிலை சர்வர்) இணைப்பதற்கான முறையைக் காட்டுகிறது. `LOCAL_MCP_COMMAND` அமைக்கப்படவில்லை என்றால் இது மெல்லிசையாக தவிர்க்கிறது, ஆகையால் நோட்புக் முழுவதும் இயங்குகிறது.

பாதுகாப்பு குறிப்புகள்: உள்ளமைந்த MCP சர்வர் உங்கள் பயனர் அனுமதிகளுடன் இயங்குகிறது. அதை ஒரு திட்ட அடைவில் மட்டுப்படுத்தவும் மற்றும் அதன் வெளிப்பாட்டுகளை செயல்படுத்துவதற்கு முன் சரிபார்க்கவும்.


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## சுருக்கம்

நீங்கள் முழுக்க உங்கள் கருவியில் இயங்கும் ஒரு பொறியியல் உதவியாளரை உருவாக்கினீர்கள்:

- **Foundry Local** ஒரு **Qwen** மாதிரியை OpenAI-க்கு இணக்கமான இடைமுகம் மூலம் வழங்கியது — ஆகையால் ஏஜென்ட் குறியீடு மேகம் வழிமுறைகளுடன் பொருந்தும்.
- **Sandboxed tools** ஏஜென்டுக்கு கோப்பு அணுகல் மற்றும் குறியீட்டு பகுப்பாய்வை திட்ட அடைவு விட்டு வெளியே செல்லாமல் வழங்கின.
- **Chroma** உள்ளூர் RAG-ஐ ஆவணங்களுக்காக வழங்கியது.
- **Local MCP** MCP சுற்றுச்சூழலையொன்றை ஆஃப்லைனில் மீண்டும் பயன்பாடாக்குவது எப்படி என்பதை காட்டியது.

எந்த நேரத்திலும் மேக நிர்ணயம் பயன்படுத்தப்படவில்லை.

### சவால்

உள்ளூர் ஏஜென்டை விரிவுபடுத்தவும்:

1. **பல MCP சர்வர்களுடன் வேலை செய்ய** — ஒரு கோப்பு அமைப்பு சர்வர் மற்றும் ஒரு git சர்வரை இணைத்து ஏஜென்ட் இரண்டிற்கிடையில் தேர்வு செய்ய அனுமதி கொடு.
2. **உள்ளூர் நினைவகத்தைப் பயன்படுத்த** — குறுகிய உரையாடல் வரலாற்றை ডিস்க்-இல் நிலைநிறுத்தி உதவியாளர் முன் நடந்த உரையாடலை நினைவில் வைத்திருக்க.
3. **உள்ளூர் பல-ஏஜென்ட் ஒத்திசைவு ஆதரவு** — இரண்டாவது உள்ளூர் ஏஜென்டை (எ.கா. விமர்சகர்) சேர்த்து இரண்டு ஏஜென்ட்கள் ஒன்றாக பணியாற்ற.

அடுத்த பாடத்தில் நீங்கள்.field நிரல் செய்யப்பட்ட AI ஏஜென்ட்களை எப்படி பாதுகாப்பது என்பதை கற்கப்போகிறீர்கள்.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
